# Plant Instance Wirings â€” Notebook

End-to-end walkthrough of the two plant-instance retrieval wirings:

- **Plant->Image** (`emb_plant2image.json`) â€” given a per-plant crop instance, retrieve full field images that contain the same or a similar plant.
- **Plant->Plant** (`emb_plant2plant.json`) â€” retrieve other instances of the same species using `instance_labels`.

Both wirings convert to the same `MetadataGroup` representation used by Image->Image, so all KPIs are available including graded `knn_metadata_ndcg`.

Run all cells top-to-bottom; all figures are interactive (Plotly).

In [8]:
import json
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from pai.ag_emb.services.evaluate import (
    run_plant2image_eval,
    run_plant2plant_eval,
)
from pai.ag_emb.services.reporting import (
    plot_cosine_similarity,
    plot_knn_confusion,
    plot_tsne,
    print_result,
)

---
## Part 1 â€” Plant->Image

`emb_plant2image.json` contains:
- **6 corn parent images** (full field), each with **2 instance crops** -> 18 embeddings total.
- **6 soy parent images** (full field), each with **2 instance crops** -> 18 embeddings total.
- 36 embeddings total across both classes.

The `instance_to_image` mapping declares which instance crops belong to which parent.  
Ground truth:
- Instance -> its parent is grade-3 (explicit positive).
- Parent -> all its instances are grade-3.
- Items sharing the same crop class (e.g. corn) across different parent groups are grade-1.

**Similarity design** â€” instances of the same parent image have cosine â‰ˆ 0.97â€“0.99 to their parent (very close, not identical). Parent images of the same class are 0.85â€“0.95 apart. Cross-class (corn vs soy) is near zero.

In [9]:
with open("emb_plant2image.json") as f:
    payload = json.load(f)

embeddings = payload["embeddings"]
instance_to_image = payload["instance_to_image"]

print(f"Total embeddings : {len(embeddings)}")
print(f"Parent images    : {len(instance_to_image)}")
print(f"Instance crops   : {sum(len(v) for v in instance_to_image.values())}")
print(f"Embedding dim    : {len(next(iter(embeddings.values())))}")
print()
print("Sample parent->instances mapping:")
for parent, insts in list(instance_to_image.items())[:2]:
    print(f"  {parent}")
    for i in insts:
        print(f"    -> {i}")

Total embeddings : 18
Parent images    : 6
Instance crops   : 12
Embedding dim    : 32

Sample parentâ†’instances mapping:
  images/corn_HB-25000SBC/220621-corn-HB-25000SBC-54e94639.png
    â†’ images/corn_HB-25000SBC/220621-corn-HB-25000SBC-54e94639-0.png
    â†’ images/corn_HB-25000SBC/220621-corn-HB-25000SBC-54e94639-1.png
  images/corn_HB-25000SBC/220622-corn-HB-25000SBC-af33e7ca.png
    â†’ images/corn_HB-25000SBC/220622-corn-HB-25000SBC-af33e7ca-0.png
    â†’ images/corn_HB-25000SBC/220622-corn-HB-25000SBC-af33e7ca-1.png


In [10]:
result_p2i = run_plant2image_eval(
    embeddings=embeddings,
    instance_to_image=instance_to_image,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
)
print_result(result_p2i)

n_items      : 18
embedding_dim: 32
classes      : ['corn', 'soybean']
k_values     : [5, 10]

â”€â”€ global_metrics â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
  pairwise cosine    : mean=0.2163  std=0.3798  (p05=-0.2109  p50=0.1431  p95=0.9811)
  centroid cosine    : mean=0.5098  std=0.0664  norm=0.5098
  intra/inter gap    : 0.6597  (intra=0.5656  inter=-0.0941)
  effective_rank     : 3.30  (ratio=0.1030  dim=32)
  uniformity         : -1.8771
  alignment          : 0.0468

  hubness@5         : mean=5.0000  std=2.4495  p95=8.0000
  hubness@10        : mean=10.0000  std=1.5986  p95=12.1500
  knn_radius@5         : mean=0.4651  std=0.0689  p05=0.3805  p95=0.5547
  knn_radius@10        : mean=-0.0005  std=0.1022  p05=-0.1009  p95=0.1528
  mean_top_k_sim@5         : mean=0.6757  std=0.0410  p05=0.6225  p95=0.7277
  mean_top_k_sim@10        : mean=0.4555  std=0.0496  p0

### Interpreting the Plant->Image KPIs

| KPI | What to look for |
|---|---|
| `knn_metadata_precision@k` | Are the top-k results explicit positives (parent/instances from the same group)? |
| `knn_metadata_ndcg@k` | Is the full grade-3 set (parent + instances) ranked above grade-1 (same class, different group)? |
| `knn_label_purity@k` | Fraction of neighbours sharing the same crop class (corn or soy). |
| `alignment` | Mean `â€–uâˆ’vâ€–Â²` across explicit positive pairs â€” lower means parent and instances are closer together. |

With the tight embeddings in this file, you should see **high metadata precision and nDCG** because instances are only 0.97â€“0.99 cosine away from their parent, while cross-class items are near zero.

In [11]:
plot_knn_confusion(result_p2i, output_path=None)
plot_cosine_similarity(embeddings, result_p2i, output_path=None)
plot_tsne(embeddings, result_p2i, dimensions=2, output_path=None)

  t-SNE 2D â€” fitting 18 samples (perplexity=4, iter=1000)...


---
## Part 2 â€” Plant->Plant

`emb_plant2plant.json` contains 21 instance crops across three classes:
- **corn** â€” 9 instances (early / medium / late growth stage)
- **soybean** â€” 6 instances (early / medium growth stage)
- **weed** â€” 6 instances (broadleaf / grass)

All instances sharing the same class label are mutual grade-3 positives â€” useful for measuring class-level retrieval quality.

In [12]:
with open("emb_plant2plant.json") as f:
    payload = json.load(f)

embeddings_p2p = payload["embeddings"]
instance_labels = payload["instance_labels"]

label_counts = Counter(instance_labels.values())
print(f"Total instances   : {len(embeddings_p2p)}")
print(f"Class distribution: {dict(label_counts)}")

Total instances   : 21
Class distribution: {'corn': 9, 'soybean': 6, 'weed': 6}


In [13]:
result_p2p = run_plant2plant_eval(
    embeddings=embeddings_p2p,
    instance_labels=instance_labels,
    k_values=[5, 10],
    sample_pairs=None,
)
print("=== Plant->Plant ===")
print_result(result_p2p)

=== Plantâ†’Plant ===
n_items      : 21
embedding_dim: 32
classes      : ['corn', 'soybean', 'weed']
k_values     : [5, 10]

â”€â”€ global_metrics â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
  pairwise cosine    : mean=0.2341  std=0.2891  (p05=-0.1349  p50=0.1945  p95=0.9000)
  centroid cosine    : mean=0.5202  std=0.0962  norm=0.5202
  intra/inter gap    : 0.4521  (intra=0.5441  inter=0.0920)
  effective_rank     : 5.26  (ratio=0.1643  dim=32)
  uniformity         : -2.2291
  alignment          : 0.9118

  hubness@5         : mean=5.0000  std=1.7457  p95=8.0000
  hubness@10        : mean=10.0000  std=2.9601  p95=16.0000
  knn_radius@5         : mean=0.3932  std=0.0731  p05=0.2921  p95=0.4937
  knn_radius@10        : mean=0.2092  std=0.0742  p05=0.1328  p95=0.3428
  mean_top_k_sim@5         : mean=0.6110  std=0.0427  p05=0.5512  p95=0.6688
  mean_top_k_sim@10        : 

In [14]:
plot_knn_confusion(result_p2p, output_path=None)
plot_cosine_similarity(embeddings_p2p, result_p2p, output_path=None)
plot_tsne(embeddings_p2p, result_p2p, dimensions=2, output_path=None)

  t-SNE 2D â€” fitting 21 samples (perplexity=3, iter=1000)...
